In [1]:
from pathlib import Path
print(Path.cwd())

/content


In [3]:
%cd /content

!git clone https://github.com/Aishwarya-93/Deepfake-Detection-KYC.git

%cd Deepfake-Detection-KYC

/content
Cloning into 'Deepfake-Detection-KYC'...
remote: Enumerating objects: 94, done.
remote: Counting objects: 100% (94/94), done.
remote: Compressing objects: 100% (75/75), done.
remote: Total 94 (delta 41), reused 57 (delta 16), pack-reused 0 (from 0)
Receiving objects: 100% (94/94), 30.79 KiB | 5.13 MiB/s, done.
Resolving deltas: 100% (41/41), done.
/content/Deepfake-Detection-KYC


In [4]:
!pwd
!ls

/content/Deepfake-Detection-KYC
data  environment.yml  outputs	      README.md  requirements.txt
docs  notebooks        presentations  reports	 src


In [5]:
!kaggle datasets download -d xdxd003/ff-c23
!unzip -q ff-c23.zip -d data/raw/

Dataset URL: https://www.kaggle.com/datasets/xdxd003/ff-c23
License(s): other
100% 16.7G/16.7G [13:25<00:00, 22.2MB/s]



In [7]:
!python src/preprocessing/read_metadata.py

Reading original.csv...
Reading Deepfakes.csv...
Reading FaceSwap.csv...
Reading Face2Face.csv...
Reading FaceShifter.csv...
Reading NeuralTextures.csv...
Reading DeepFakeDetection.csv...

Successfully merged all CSV files.
Metadata cleaned successfully.

Metadata saved successfully!
Location: /content/Deepfake-Detection-KYC/data/metadata/master_metadata.csv

Metadata Engineering Completed Successfully!


In [8]:
from src.preprocessing.validate_metadata import *

metadata = load_metadata()

metadata.head()

Loaded 7000 records.


,File Path,Label,Frame Count,Width,Height
0,original/000.mp4,REAL,396,640,480
1,original/001.mp4,REAL,460,1280,720
2,original/002.mp4,REAL,693,1280,720
3,original/003.mp4,REAL,303,640,480
4,original/004.mp4,REAL,309,1280,720


In [9]:
import importlib
import src.preprocessing.validate_metadata as vm

importlib.reload(vm)

<module 'src.preprocessing.validate_metadata' from '/content/Deepfake-Detection-KYC/src/preprocessing/validate_metadata.py'>

In [10]:
vm.check_missing_values(metadata)


Missing Values:
File Path      0
Label          0
Frame Count    0
Width          0
Height         0
dtype: int64


,0
File Path,0
Label,0
Frame Count,0
Width,0
Height,0


In [12]:
# ==========================================================
# Duplicate Rows
# ==========================================================

def check_duplicates(df):
    """
    Check for duplicate rows in the metadata.
    """

    duplicate_count = df.duplicated().sum()

    print("\n========== Duplicate Rows ==========")
    print(f"Duplicate rows found: {duplicate_count}")

    return duplicate_count

In [16]:
def generate_validation_report(df):
    """
    Generate a summary report of metadata validation.
    """

    missing_values = df.isnull().sum().sum()
    duplicate_rows = df.duplicated().sum()

    class_distribution = df["Label"].value_counts()

    print("\n===================================")
    print("     Metadata Validation Report")
    print("===================================")

    print(f"Total Videos      : {len(df)}")
    print(f"Missing Values    : {missing_values}")
    print(f"Duplicate Rows    : {duplicate_rows}")

    print("\nClass Distribution:")

    for label, count in class_distribution.items():
        print(f"{label:5}: {count}")
    status = "PASS"

    if missing_values > 0 or duplicate_rows > 0:
        status = "FAIL"

    print(f"\nValidation Status : {status}")

In [26]:
generate_validation_report(metadata)


     Metadata Validation Report
Total Videos      : 7000
Missing Values    : 0
Duplicate Rows    : 0

Class Distribution:
FAKE : 6000
REAL : 1000

Validation Status : PASS


In [18]:
# ==========================================================
# Frame Statistics
# ==========================================================

def frame_statistics(df):
    """
    Display statistics for the Frame Count column.
    """

    print("\n========== Frame Statistics ==========")

    statistics = df["Frame Count"].describe()

    print(statistics)

    return statistics

In [19]:
frame_statistics(metadata)


========== Frame Statistics ==========
count    7000.000000
mean      511.797143
std       229.865219
min         5.000000
25%       346.000000
50%       444.000000
75%       594.000000
max      1814.000000
Name: Frame Count, dtype: float64


,Frame Count
count,7000.000000
mean,511.797143
std,229.865219
min,5.000000
25%,346.000000
50%,444.000000
75%,594.000000
max,1814.000000


In [20]:
# ==========================================================
# Resolution Statistics
# ==========================================================

def resolution_statistics(df):
    """
    Display the distribution of video resolutions.
    """

    print("\n========== Resolution Statistics ==========")

    resolution = (
        df["Width"].astype(str)
        + " x "
        + df["Height"].astype(str)
    )

    counts = resolution.value_counts()

    print(counts)

    return counts

In [21]:
resolution_statistics(metadata)


========== Resolution Statistics ==========
1280 x 720     1950
1920 x 1080    1738
640 x 480      1724
854 x 480       392
656 x 480       264
832 x 480       204
600 x 480       192
720 x 480       108
576 x 480       104
654 x 480        80
704 x 480        56
960 x 720        50
1440 x 1080      18
848 x 480        12
598 x 480         8
658 x 480         8
642 x 480         8
1264 x 720        8
800 x 480         8
1248 x 720        6
1280 x 718        6
272 x 480         4
852 x 480         4
644 x 480         4
586 x 480         4
454 x 480         4
636 x 480         4
602 x 480         4
632 x 480         4
810 x 480         4
1274 x 720        4
712 x 480         4
978 x 720         4
608 x 480         4
448 x 480         2
256 x 480         2
Name: count, dtype: int64


,count
1280 x 720,1950
1920 x 1080,1738
640 x 480,1724
854 x 480,392
656 x 480,264
832 x 480,204
600 x 480,192
720 x 480,108
576 x 480,104
654 x 480,80


In [22]:
# ==========================================================
# Video Path Validation
# ==========================================================

from pathlib import Path

DATASET_PATH = PROJECT_ROOT / "data" / "raw" / "FaceForensics++_C23"


def check_video_paths(df):
    """
    Verify that every video listed in the metadata exists.
    """

    print("\n========== Video Path Validation ==========")

    missing_files = []

    for file_path in df["File Path"]:

        video_path = DATASET_PATH / file_path

        if not video_path.exists():
            missing_files.append(file_path)

    print(f"Missing files : {len(missing_files)}")

    return missing_files

In [27]:
frame_stats = frame_statistics(metadata)

resolution_counts = resolution_statistics(metadata)

missing_files = check_video_paths(metadata)

print(f"Missing Files : {len(missing_files)}")
print(f"Average Frames : {frame_stats['mean']:.2f}")
print(f"Video Resolutions Found : {len(resolution_counts)}")


========== Frame Statistics ==========
count    7000.000000
mean      511.797143
std       229.865219
min         5.000000
25%       346.000000
50%       444.000000
75%       594.000000
max      1814.000000
Name: Frame Count, dtype: float64

========== Resolution Statistics ==========
1280 x 720     1950
1920 x 1080    1738
640 x 480      1724
854 x 480       392
656 x 480       264
832 x 480       204
600 x 480       192
720 x 480       108
576 x 480       104
654 x 480        80
704 x 480        56
960 x 720        50
1440 x 1080      18
848 x 480        12
598 x 480         8
658 x 480         8
642 x 480         8
1264 x 720        8
800 x 480         8
1248 x 720        6
1280 x 718        6
272 x 480         4
852 x 480         4
644 x 480         4
586 x 480         4
454 x 480         4
636 x 480         4
602 x 480         4
632 x 480         4
810 x 480         4
1274 x 720        4
712 x 480         4
978 x 720         4
608 x 480         4
448 x 480         2
256 x 480    

In [28]:
def generate_validation_report(df):

    missing_values = df.isnull().sum().sum()
    duplicate_rows = df.duplicated().sum()
    class_distribution = df["Label"].value_counts()

    frame_stats = frame_statistics(df)
    resolution_counts = resolution_statistics(df)
    missing_files = check_video_paths(df)

    status = "PASS"

    if (
        missing_values > 0
        or duplicate_rows > 0
        or len(missing_files) > 0
    ):
        status = "FAIL"

    print("\n===================================")
    print("     Metadata Validation Report")
    print("===================================")

    print(f"Total Videos          : {len(df)}")
    print(f"Missing Values        : {missing_values}")
    print(f"Duplicate Rows        : {duplicate_rows}")
    print(f"Missing Files         : {len(missing_files)}")
    print(f"Average Frames        : {frame_stats['mean']:.2f}")
    print(f"Resolution Types      : {len(resolution_counts)}")

    print("\nClass Distribution:")
    for label, count in class_distribution.items():
        print(f"{label:5}: {count}")

    print(f"\nValidation Status : {status}")

In [29]:
generate_validation_report(metadata)


========== Frame Statistics ==========
count    7000.000000
mean      511.797143
std       229.865219
min         5.000000
25%       346.000000
50%       444.000000
75%       594.000000
max      1814.000000
Name: Frame Count, dtype: float64

========== Resolution Statistics ==========
1280 x 720     1950
1920 x 1080    1738
640 x 480      1724
854 x 480       392
656 x 480       264
832 x 480       204
600 x 480       192
720 x 480       108
576 x 480       104
654 x 480        80
704 x 480        56
960 x 720        50
1440 x 1080      18
848 x 480        12
598 x 480         8
658 x 480         8
642 x 480         8
1264 x 720        8
800 x 480         8
1248 x 720        6
1280 x 718        6
272 x 480         4
852 x 480         4
644 x 480         4
586 x 480         4
454 x 480         4
636 x 480         4
602 x 480         4
632 x 480         4
810 x 480         4
1274 x 720        4
712 x 480         4
978 x 720         4
608 x 480         4
448 x 480         2
256 x 480    